# Step 1: Environment Configuration and Library Import

In [ ]:
pip install yfinance scikit-learn tqdm matplotlib

In [ ]:
!pip install --no-deps git+https://github.com/Blealtan/efficient-kan.git

In [ ]:
pip install efficient-kan

In [ ]:
import math

try:
    from efficient_kan import KAN
    HAS_KAN = True
except ImportError:
    HAS_KAN = False
    print("Warning: efficient-kan library not found. If not installed, KAN models will be skipped. Please run: %pip install efficient-kan")

In [ ]:
!pip install --upgrade torch torchvision torchaudio

# Step 2: Data Acquisition & Inspection

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import math

# ------------------------------------------
# 1. Data Acquisition and Feature Engineering
# ------------------------------------------
def get_optimized_timeseries_dataloaders(ticker="^GSPC", lookback=60, batch_size=32):
    print(f"Fetching historical data for [{ticker}] from Yahoo Finance...")
    data = yf.download(ticker, period="10y", progress=False)

    df = pd.DataFrame()
    df['Close'] = data['Close']
    df['Open'] = data['Open']
    df['High'] = data['High']
    df['Low'] = data['Low']
    df['Volume'] = data['Volume']

    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna()

    feature_cols = ['LogReturn', 'Open', 'High', 'Low', 'Volume', 'MA5', 'MA20']
    features = df[feature_cols].values
    target = df[['LogReturn']].values

    split_idx = int(len(features) * 0.8)
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()

    train_features = feature_scaler.fit_transform(features[:split_idx])
    train_target = target_scaler.fit_transform(target[:split_idx])
    test_features = feature_scaler.transform(features[split_idx:])
    test_target = target_scaler.transform(target[split_idx:])

    scaled_features = np.vstack((train_features, test_features))
    scaled_target = np.vstack((train_target, test_target))

    X, y = [], []
    for i in range(len(scaled_features) - lookback):
        X.append(scaled_features[i:(i + lookback), :])
        y.append(scaled_target[i + lookback, 0])

    X = torch.tensor(np.array(X), dtype=torch.float32)
    y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(1)

    actual_split = split_idx - lookback
    X_train, y_train = X[:actual_split], y[:actual_split]
    X_test, y_test = X[actual_split:], y[actual_split:]

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    prev_close_prices = df['Close'].values[split_idx - 1 : -1]
    actual_close_prices = df['Close'].values[split_idx : ]

    print(f"Data preparation complete! Training set: {len(X_train)} samples, Test set: {len(X_test)} samples. Window length: {lookback}")
    return train_loader, test_loader, target_scaler, prev_close_prices, actual_close_prices, len(feature_cols)

# ------------------------------------------
# 2. Model Class Definitions
# ------------------------------------------
SEQ_LEN = 60
FEATURE_DIM = 7
EMBED_DIM = 64
HIDDEN_DIM = 256
OUTPUT_DIM = 1

# ------------------------------------------
# TS_MLP
# ------------------------------------------
class TS_MLP(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, feature_dim=FEATURE_DIM):
        super().__init__()
        self.flatten = nn.Flatten()
        # 420 -> 128 -> 256 -> 64 -> 1
        self.fc1 = nn.Linear(seq_len * feature_dim, 128)
        self.fc2 = nn.Linear(128, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc_out = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(F.relu(self.fc2(x)))
        x = F.relu(self.fc3(x))
        return self.fc_out(x)

# ------------------------------------------
# TS_CNN
# ------------------------------------------
class TS_CNN(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, feature_dim=FEATURE_DIM):
        super().__init__()
        self.conv1 = nn.Conv1d(feature_dim, 128, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

# ------------------------------------------
# TS_Transformer
# ------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TS_Transformer(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, feature_dim=FEATURE_DIM, embed_dim=EMBED_DIM, num_heads=4):
        super().__init__()
        self.input_proj = nn.Linear(feature_dim, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True, dropout=0.2, dim_feedforward=HIDDEN_DIM
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        return self.fc(x.mean(dim=1))

# ------------------------------------------
# TS_Mamba
# ------------------------------------------
class SimpleMambaBlock(nn.Module):
    def __init__(self, d_model=EMBED_DIM, d_state=16):
        super().__init__()
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=4, padding=3, groups=d_model)
        self.x_proj = nn.Linear(d_model, d_state * 2 + d_model)
        self.dt_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_in.transpose(1, 2))[:, :, :L].transpose(1, 2))
        x_proj = self.x_proj(x_conv)
        delta, A, B_val = torch.split(x_proj, [D, 16, 16], dim=-1)
        return self.out_proj((x_conv * F.softplus(self.dt_proj(delta))) * F.silu(z))

class TS_Mamba(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, feature_dim=FEATURE_DIM, embed_dim=EMBED_DIM, num_layers=4):
        super().__init__()
        self.proj = nn.Linear(feature_dim, embed_dim)
        self.layers = nn.ModuleList([SimpleMambaBlock(embed_dim) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(num_layers)])
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        x = self.proj(x)
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + layer(x))
        return self.fc(x.mean(dim=1))

# ------------------------------------------
# TS_KAN
# ------------------------------------------
class TS_KAN(nn.Module):
    def __init__(self, seq_len=SEQ_LEN, feature_dim=FEATURE_DIM):
        super().__init__()
        self.flatten = nn.Flatten()
        if HAS_KAN:

            self.kan = KAN(
                [seq_len * feature_dim, 26, 16, 1],
                grid_size=5,
                spline_order=3
            )
        else:
            self.kan = None

    def forward(self, x):
        x = self.flatten(x)
        return self.kan(x) if self.kan else torch.zeros(x.shape[0], 1)

# ------------------------------------------
# 3. Unified Training and Evaluation Functions
# ------------------------------------------
def evaluate_model(model, test_loader, target_scaler, device):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            predictions.extend(model(X_batch.to(device)).cpu().numpy())
            actuals.extend(y_batch.numpy())

    predictions = target_scaler.inverse_transform(predictions).flatten()
    actuals = target_scaler.inverse_transform(actuals).flatten()
    return predictions, actuals, np.mean((predictions - actuals)**2), np.mean(np.sign(predictions)==np.sign(actuals))*100

def train_and_eval(model_name, model, loaders, target_scaler, device, epochs=30):
    print(f"\n{'='*50}\nStarting training: [{model_name}]")
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    model.train()

    epoch_loop = tqdm(range(epochs), desc=f"Training Progress", colour='green')
    for epoch in epoch_loop:
        train_loss = 0
        for X_batch, y_batch in loaders['train']:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = nn.MSELoss()(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        epoch_loop.set_postfix(loss=f"{train_loss/len(loaders['train']):.6f}")

    preds, _, mse, da = evaluate_model(model, loaders['test'], target_scaler, device)
    print(f"[{model_name}] Training complete! Test set MSE: {mse:.6f} | Directional Accuracy (DA): {da:.2f}%")
    return preds, mse, da

# ------------------------------------------
# 4. Initialize Data
# ------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Current compute device: {device}")

lookback_window = 60
train_loader, test_loader, target_scaler, prev_close_prices, actual_close_prices, feature_dim = get_optimized_timeseries_dataloaders(lookback=lookback_window)
loaders = {'train': train_loader, 'test': test_loader}

all_raw_preds = {}
all_da = {}
print("\nPhase 1: Environment and data preparation is ready, please run the next code block.")

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def format_params(num_params):
    return f"{num_params / 1e6:.4f} M"

print("Initializing models to calculate parameter counts...")
models_to_test = {
    "TS_MLP": TS_MLP(),
    "TS_CNN": TS_CNN(),
    "TS_Transformer": TS_Transformer(),
    "TS_Mamba": TS_Mamba(),
}

if HAS_KAN:
    models_to_test["TS_KAN"] = TS_KAN()

print(f"\n{'='*55}")
print(f"Exact Model Parameter Count Statistics (Under current hyperparameters)")
print(f"{'='*55}")
print(f"{'Model Name':<18} | {'Unit (M)':<12} | {'Exact Count'}")
print(f"{'-'*55}")

for name, model in models_to_test.items():
    params = count_parameters(model)
    print(f"  {name:<16} | {format_params(params):<12} | {params:,}")

print(f"{'='*55}")

# Step 3: Model Training

In [ ]:
# ==========================================
# Training of the first four models
# ==========================================
print(">>> Starting the training pipeline for the four classic models...")

models_to_run = {
    "TS_MLP": TS_MLP(seq_len=lookback_window, feature_dim=feature_dim),
    "TS_CNN": TS_CNN(seq_len=lookback_window, feature_dim=feature_dim),
    "TS_Transformer": TS_Transformer(seq_len=lookback_window, feature_dim=feature_dim),
    "TS_Mamba": TS_Mamba(seq_len=lookback_window, feature_dim=feature_dim)
}

for name, model in models_to_run.items():
    preds_log_returns, mse, da = train_and_eval(name, model, loaders, target_scaler, device, epochs=30)
    all_raw_preds[name] = preds_log_returns
    all_da[name] = da

print("\nPhase 2: Training of the four classic models complete, please run the next code block.")

In [ ]:
# ==========================================
# KAN Model Training
# ==========================================
if HAS_KAN:
    print(">>> Starting KAN model training...")
    kan_model = TS_KAN(seq_len=lookback_window, feature_dim=feature_dim)
    preds_log_returns, mse, da = train_and_eval("TS_KAN", kan_model, loaders, target_scaler, device, epochs=30)
    all_raw_preds["TS_KAN"] = preds_log_returns
    all_da["TS_KAN"] = da
else:
    print("HAS_KAN is False, skipping KAN model and directly plotting charts for other models...")

# Step 4: Evaluation & Visual Comparison

In [ ]:
all_predictions = {}
for name, preds_log_returns in all_raw_preds.items():
    reconstructed_prices = []
    for i, log_ret in enumerate(preds_log_returns):
        pred_price = prev_close_prices[i] * np.exp(log_ret)
        reconstructed_prices.append(pred_price)
    all_predictions[name] = reconstructed_prices

plt.figure(figsize=(14, 7))
plot_len = min(200, len(actual_close_prices))

plt.plot(actual_close_prices[-plot_len:], label="True Market Price (S&P 500)", color='black', linewidth=2.5, zorder=10)

color_map = {
    "TS_MLP": '#9E9E9E',
    "TS_CNN": '#1976D2',
    "TS_Transformer": '#4CAF50',
    "TS_Mamba": '#E53935',
    "TS_KAN": '#9C27B0'
}

for name, preds in all_predictions.items():
    c = color_map.get(name, '#F39C12')
    plt.plot(preds[-plot_len:], label=f"{name} (DA: {all_da[name]:.2f}%)", color=c, alpha=0.85, linewidth=1.5)

plt.title("Financial Time-Series: 1-Step Ahead Price Prediction Comparison", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Days (Test Set, Last 200 Days)", fontsize=12)
plt.ylabel("Index Price (USD)", fontsize=12)
plt.legend(loc='upper left', frameon=True, shadow=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()